# Notebook 03 — Feature Extraction & Windowing
Prepares two representations used by downstream models:
1. **Windowed raw arrays** `X_raw` — input for 1D-CNN
2. **Hand-crafted feature matrix** `X_feat` — input for RF/SVM/XGBoost

Also applies harmonization (resampling, low-pass filter).


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/fall-detection-generalization'
    %pip install -q numpy pandas scipy scikit-learn tqdm
    sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    REPO_ROOT = str(pathlib.Path('..').resolve())

DATA_ROOT = REPO_ROOT + '/data/raw'
RESULTS   = REPO_ROOT + '/results'

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

from src.data.sisfall import load_sisfall
from src.data.harmonize import harmonize_datasets, TARGET_FS
from src.features.extract import extract_dataset_features

Path(RESULTS).mkdir(exist_ok=True)

## 1. Load SisFall and harmonize (resample to 50 Hz, low-pass filter)

In [ ]:
trials = load_sisfall(Path(DATA_ROOT) / 'sisfall')
print(f'Loaded {len(trials)} trials')

X_raw, y, subjects, age_groups = harmonize_datasets(trials, sisfall_fs=200, target_fs=TARGET_FS)
print(f'Windows: {X_raw.shape}  →  {X_raw.shape[0]} windows × {X_raw.shape[1]} timesteps × {X_raw.shape[2]} channels')
print(f'Class balance: {np.bincount(y)} (ADL / Fall)')

## 2. Save raw windowed arrays (for CNN)

In [ ]:
np.save(f'{RESULTS}/X_raw.npy',      X_raw)
np.save(f'{RESULTS}/y.npy',           y)
np.save(f'{RESULTS}/subjects.npy',    np.array(subjects))
np.save(f'{RESULTS}/age_groups.npy',  np.array(age_groups))
print('Raw arrays saved.')

## 3. Extract hand-crafted features (for classical ML)

In [ ]:
print('Extracting features ... (this takes a few minutes)')
df_features = extract_dataset_features(X_raw, y, subjects, age_groups, fs=TARGET_FS)
print(f'Feature matrix: {df_features.shape}')
df_features.head()

In [ ]:
df_features.to_parquet(f'{RESULTS}/features.parquet', index=False)
print('Feature matrix saved to results/features.parquet')

## 4. Class distribution and subject split summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

print(df_features.groupby(['age_group', 'label']).size().rename('windows').to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_features.groupby(['age_group', 'label']).size().unstack().plot(
    kind='bar', ax=axes[0], rot=0, color=['#4c72b0', '#dd8452'])
axes[0].set_title('Window counts by age group and label')
axes[0].legend(['ADL (0)', 'Fall (1)'])

df_features.groupby('subject_id').size().sort_values().plot(
    kind='bar', ax=axes[1], rot=90, fontsize=7)
axes[1].set_title('Windows per subject')
plt.tight_layout()
plt.savefig(f'{RESULTS}/windowing_summary.png', bbox_inches='tight')
plt.show()

## 5. Feature variance check — drop near-zero-variance features


In [ ]:
from sklearn.feature_selection import VarianceThreshold

meta_cols = ['label', 'subject_id', 'age_group']
feat_cols = [c for c in df_features.columns if c not in meta_cols]

X_check = df_features[feat_cols].values
sel = VarianceThreshold(threshold=1e-6)
sel.fit(X_check)
kept = [feat_cols[i] for i, v in enumerate(sel.get_support()) if v]
dropped = set(feat_cols) - set(kept)
print(f'Dropped {len(dropped)} near-zero-variance features; keeping {len(kept)}')
if dropped:
    print('Dropped:', dropped)

df_final = df_features[meta_cols + kept]
df_final.to_parquet(f'{RESULTS}/features_filtered.parquet', index=False)
print(f'Final feature matrix: {df_final.shape}')